# Day 2 — Databricks Medallion Pipeline

Pipeline:

Raw → Bronze → Silver → Gold → Power BI

Source table: `marketing_interactions_raw`

**Load the raw table**

In [0]:
raw_df = spark.table("default.marketing_interactions_raw")

display(raw_df.limit(10))

ad_spend,campaign_id,clicks,customer_age,customer_gender,customer_id,customer_region,device_type,discount_percent,impressions,interaction_date,marketing_channel,orders,product_category,product_id,revenue,unit_price,units_sold,website_visits
20.29,CMP002,0,25,Male,C0474,Balochistan,Tablet,10.0,2,2025-03-02,Social Media,0,Accessories,P005,0.0,2500.0,0,0
56.26,CMP002,0,33,Male,C0831,Balochistan,Tablet,15.0,5,2025-07-09,Social Media,0,Accessories,P005,0.0,2500.0,0,0
17.82,CMP004,0,28,Female,C0548,Balochistan,Tablet,25.0,2,2025-01-08,Display,0,Accessories,P005,0.0,2500.0,0,0
24.01,CMP004,0,44,Male,C0336,Punjab,Mobile,5.0,4,2025-01-03,Display,0,Footwear,P003,0.0,8000.0,0,0
62.89,CMP004,0,45,Female,C0487,Islamabad,Tablet,10.0,10,2025-07-23,Display,0,Electronics,P001,0.0,50000.0,0,0
218.94,CMP003,0,31,Female,C0854,Khyber Pakhtunkhwa,Mobile,15.0,10,2025-01-13,Search,0,Electronics,P001,0.0,50000.0,0,0
10.34,CMP001,1,58,Male,C0050,Sindh,Mobile,30.0,1,2025-12-08,Email,1,Electronics,P001,35000.0,50000.0,1,1
137.95,CMP003,1,40,Male,C0576,Khyber Pakhtunkhwa,Tablet,0.0,9,2025-11-16,Search,0,Footwear,P003,0.0,8000.0,0,1
55.98,CMP002,0,54,Female,C0570,Sindh,Desktop,10.0,4,2025-12-02,Social Media,0,Accessories,P005,0.0,2500.0,0,0
76.6,CMP001,0,63,Male,C0853,Punjab,Tablet,15.0,9,2025-12-15,Email,0,Accessories,P005,0.0,2500.0,0,0


**Inspect the dataset**

In [0]:
print(f"Total rows: {raw_df.count():,}")
print(f"Total columns: {len(raw_df.columns)}")

raw_df.printSchema()

Total rows: 10,000
Total columns: 19
root
 |-- ad_spend: double (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- clicks: long (nullable = true)
 |-- customer_age: long (nullable = true)
 |-- customer_gender: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_region: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- impressions: long (nullable = true)
 |-- interaction_date: date (nullable = true)
 |-- marketing_channel: string (nullable = true)
 |-- orders: long (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- revenue: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- units_sold: long (nullable = true)
 |-- website_visits: long (nullable = true)



In [0]:
display(raw_df.limit(10))

ad_spend,campaign_id,clicks,customer_age,customer_gender,customer_id,customer_region,device_type,discount_percent,impressions,interaction_date,marketing_channel,orders,product_category,product_id,revenue,unit_price,units_sold,website_visits
20.29,CMP002,0,25,Male,C0474,Balochistan,Tablet,10.0,2,2025-03-02,Social Media,0,Accessories,P005,0.0,2500.0,0,0
56.26,CMP002,0,33,Male,C0831,Balochistan,Tablet,15.0,5,2025-07-09,Social Media,0,Accessories,P005,0.0,2500.0,0,0
17.82,CMP004,0,28,Female,C0548,Balochistan,Tablet,25.0,2,2025-01-08,Display,0,Accessories,P005,0.0,2500.0,0,0
24.01,CMP004,0,44,Male,C0336,Punjab,Mobile,5.0,4,2025-01-03,Display,0,Footwear,P003,0.0,8000.0,0,0
62.89,CMP004,0,45,Female,C0487,Islamabad,Tablet,10.0,10,2025-07-23,Display,0,Electronics,P001,0.0,50000.0,0,0
218.94,CMP003,0,31,Female,C0854,Khyber Pakhtunkhwa,Mobile,15.0,10,2025-01-13,Search,0,Electronics,P001,0.0,50000.0,0,0
10.34,CMP001,1,58,Male,C0050,Sindh,Mobile,30.0,1,2025-12-08,Email,1,Electronics,P001,35000.0,50000.0,1,1
137.95,CMP003,1,40,Male,C0576,Khyber Pakhtunkhwa,Tablet,0.0,9,2025-11-16,Search,0,Footwear,P003,0.0,8000.0,0,1
55.98,CMP002,0,54,Female,C0570,Sindh,Desktop,10.0,4,2025-12-02,Social Media,0,Accessories,P005,0.0,2500.0,0,0
76.6,CMP001,0,63,Male,C0853,Punjab,Tablet,15.0,9,2025-12-15,Email,0,Accessories,P005,0.0,2500.0,0,0


## Bronze Layer

The Bronze table preserves the raw data and adds ingestion metadata:

- `ingestion_timestamp`
- `source_name`
- `load_date`

In [0]:
from pyspark.sql import functions as F

bronze_df = (
    raw_df
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_name", F.lit("synthetic_marketing_data"))
    .withColumn("load_date", F.current_date())
)

display(bronze_df.limit(10))

ad_spend,campaign_id,clicks,customer_age,customer_gender,customer_id,customer_region,device_type,discount_percent,impressions,interaction_date,marketing_channel,orders,product_category,product_id,revenue,unit_price,units_sold,website_visits,ingestion_timestamp,source_name,load_date
20.29,CMP002,0,25,Male,C0474,Balochistan,Tablet,10.0,2,2025-03-02,Social Media,0,Accessories,P005,0.0,2500.0,0,0,2026-08-21T11:15:20.032Z,synthetic_marketing_data,2026-08-21
56.26,CMP002,0,33,Male,C0831,Balochistan,Tablet,15.0,5,2025-07-09,Social Media,0,Accessories,P005,0.0,2500.0,0,0,2026-08-21T11:15:20.032Z,synthetic_marketing_data,2026-08-21
17.82,CMP004,0,28,Female,C0548,Balochistan,Tablet,25.0,2,2025-01-08,Display,0,Accessories,P005,0.0,2500.0,0,0,2026-08-21T11:15:20.032Z,synthetic_marketing_data,2026-08-21
24.01,CMP004,0,44,Male,C0336,Punjab,Mobile,5.0,4,2025-01-03,Display,0,Footwear,P003,0.0,8000.0,0,0,2026-08-21T11:15:20.032Z,synthetic_marketing_data,2026-08-21
62.89,CMP004,0,45,Female,C0487,Islamabad,Tablet,10.0,10,2025-07-23,Display,0,Electronics,P001,0.0,50000.0,0,0,2026-08-21T11:15:20.032Z,synthetic_marketing_data,2026-08-21
218.94,CMP003,0,31,Female,C0854,Khyber Pakhtunkhwa,Mobile,15.0,10,2025-01-13,Search,0,Electronics,P001,0.0,50000.0,0,0,2026-08-21T11:15:20.032Z,synthetic_marketing_data,2026-08-21
10.34,CMP001,1,58,Male,C0050,Sindh,Mobile,30.0,1,2025-12-08,Email,1,Electronics,P001,35000.0,50000.0,1,1,2026-08-21T11:15:20.032Z,synthetic_marketing_data,2026-08-21
137.95,CMP003,1,40,Male,C0576,Khyber Pakhtunkhwa,Tablet,0.0,9,2025-11-16,Search,0,Footwear,P003,0.0,8000.0,0,1,2026-08-21T11:15:20.032Z,synthetic_marketing_data,2026-08-21
55.98,CMP002,0,54,Female,C0570,Sindh,Desktop,10.0,4,2025-12-02,Social Media,0,Accessories,P005,0.0,2500.0,0,0,2026-08-21T11:15:20.032Z,synthetic_marketing_data,2026-08-21
76.6,CMP001,0,63,Male,C0853,Punjab,Tablet,15.0,9,2025-12-15,Email,0,Accessories,P005,0.0,2500.0,0,0,2026-08-21T11:15:20.032Z,synthetic_marketing_data,2026-08-21


In [0]:
bronze_df.printSchema()

root
 |-- ad_spend: double (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- clicks: long (nullable = true)
 |-- customer_age: long (nullable = true)
 |-- customer_gender: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_region: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- impressions: long (nullable = true)
 |-- interaction_date: date (nullable = true)
 |-- marketing_channel: string (nullable = true)
 |-- orders: long (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- revenue: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- units_sold: long (nullable = true)
 |-- website_visits: long (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- source_name: string (nullable = false)
 |-- load_date: date (nullable = false)



In [0]:
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("default.bronze_marketing_interactions")
)

print("Bronze table created successfully.")

Bronze table created successfully.


In [0]:
saved_bronze_df = spark.table(
    "default.bronze_marketing_interactions"
)

raw_count = raw_df.count()
bronze_count = saved_bronze_df.count()

print(f"Raw rows: {raw_count:,}")
print(f"Bronze rows: {bronze_count:,}")

assert raw_count == bronze_count, (
    "Validation failed: Raw and Bronze row counts differ."
)

print("Bronze validation passed.")

Raw rows: 10,000
Bronze rows: 10,000
Bronze validation passed.


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(ingestion_timestamp) AS rows_with_ingestion_timestamp,
    COUNT(source_name) AS rows_with_source_name,
    COUNT(load_date) AS rows_with_load_date,
    COUNT(DISTINCT source_name) AS distinct_sources
FROM default.bronze_marketing_interactions;

total_rows,rows_with_ingestion_timestamp,rows_with_source_name,rows_with_load_date,distinct_sources
10000,10000,10000,10000,1


**Silver**

In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table("default.bronze_marketing_interactions")

business_columns = [
    column for column in bronze_df.columns
    if column not in ["ingestion_timestamp", "source_name", "load_date"]
]

total_rows = bronze_df.count()
duplicate_rows = total_rows - bronze_df.dropDuplicates(business_columns).count()

print(f"Total rows: {total_rows:,}")
print(f"Exact duplicate business rows: {duplicate_rows:,}")

Total rows: 10,000
Exact duplicate business rows: 0


**Check missing values**

In [0]:
null_counts = bronze_df.select([
    F.sum(F.col(column).isNull().cast("int")).alias(column)
    for column in bronze_df.columns
])

display(null_counts)

ad_spend,campaign_id,clicks,customer_age,customer_gender,customer_id,customer_region,device_type,discount_percent,impressions,interaction_date,marketing_channel,orders,product_category,product_id,revenue,unit_price,units_sold,website_visits,ingestion_timestamp,source_name,load_date
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


**Check funnel and revenue rules**

In [0]:
expected_revenue = (
    F.col("unit_price")
    * F.col("units_sold")
    * (1 - F.col("discount_percent") / 100)
)

quality_report = bronze_df.agg(
    F.sum(
        (
            (F.col("impressions") < 0) |
            (F.col("clicks") < 0) |
            (F.col("clicks") > F.col("impressions"))
        ).cast("int")
    ).alias("invalid_click_funnel"),

    F.sum(
        (
            (F.col("website_visits") < 0) |
            (F.col("website_visits") > F.col("clicks"))
        ).cast("int")
    ).alias("invalid_visit_funnel"),

    F.sum(
        (
            (F.col("orders") < 0) |
            (F.col("orders") > F.col("website_visits"))
        ).cast("int")
    ).alias("invalid_order_funnel"),

    F.sum(
        (
            ((F.col("orders") == 0) & (F.col("units_sold") != 0)) |
            ((F.col("orders") > 0) & (F.col("units_sold") <= 0))
        ).cast("int")
    ).alias("invalid_units"),

    F.sum(
        (
            (F.col("discount_percent") < 0) |
            (F.col("discount_percent") > 100)
        ).cast("int")
    ).alias("invalid_discount"),

    F.sum(
        (
            (F.col("ad_spend") < 0) |
            (F.col("unit_price") < 0) |
            (F.col("revenue") < 0)
        ).cast("int")
    ).alias("negative_financial_values"),

    F.sum(
        (
            F.abs(F.col("revenue") - expected_revenue) > 0.01
        ).cast("int")
    ).alias("invalid_revenue")
)

display(quality_report)

invalid_click_funnel,invalid_visit_funnel,invalid_order_funnel,invalid_units,invalid_discount,negative_financial_values,invalid_revenue
0,0,0,0,0,0,0


**Check Dublicates**

In [0]:
# Exclude Bronze metadata when checking business-data duplicates
business_columns = [
    column for column in bronze_df.columns
    if column not in [
        "ingestion_timestamp",
        "source_name",
        "load_date"
    ]
]

total_rows = bronze_df.count()

unique_rows = (
    bronze_df
    .dropDuplicates(business_columns)
    .count()
)

duplicate_rows = total_rows - unique_rows

print(f"Total Bronze rows: {total_rows:,}")
print(f"Unique business rows: {unique_rows:,}")
print(f"Exact duplicate business rows: {duplicate_rows:,}")

if duplicate_rows == 0:
    print("Duplicate validation passed.")
else:
    print(f"Warning: {duplicate_rows:,} duplicate rows were found.")

Total Bronze rows: 10,000
Unique business rows: 10,000
Exact duplicate business rows: 0
Duplicate validation passed.


In [0]:
duplicate_records = (
    bronze_df
    .groupBy(*business_columns)
    .count()
    .filter(F.col("count") > 1)
)

display(duplicate_records)

ad_spend,campaign_id,clicks,customer_age,customer_gender,customer_id,customer_region,device_type,discount_percent,impressions,interaction_date,marketing_channel,orders,product_category,product_id,revenue,unit_price,units_sold,website_visits,count


**Silver DataFrame:**

In [0]:
from pyspark.sql import functions as F

business_columns = [
    column for column in bronze_df.columns
    if column not in [
        "ingestion_timestamp",
        "source_name",
        "load_date"
    ]
]

expected_revenue = (
    F.col("unit_price")
    * F.col("units_sold")
    * (1 - F.col("discount_percent") / 100)
)

silver_df = (
    bronze_df

    # Remove exact duplicate business records
    .dropDuplicates(business_columns)

    # Remove records missing essential identifiers or dates
    .dropna(subset=[
        "campaign_id",
        "customer_id",
        "product_id",
        "interaction_date"
    ])

    # Standardize IDs
    .withColumn("campaign_id", F.upper(F.trim("campaign_id")))
    .withColumn("customer_id", F.upper(F.trim("customer_id")))
    .withColumn("product_id", F.upper(F.trim("product_id")))

    # Standardize descriptive text
    .withColumn("customer_gender", F.initcap(F.trim("customer_gender")))
    .withColumn("customer_region", F.initcap(F.trim("customer_region")))
    .withColumn("device_type", F.initcap(F.trim("device_type")))
    .withColumn("marketing_channel", F.initcap(F.trim("marketing_channel")))
    .withColumn("product_category", F.initcap(F.trim("product_category")))

    # Keep only valid records
    .filter(
        (F.col("impressions") >= 0) &
        (F.col("clicks").between(0, F.col("impressions"))) &
        (F.col("website_visits").between(0, F.col("clicks"))) &
        (F.col("orders").between(0, F.col("website_visits"))) &
        (F.col("discount_percent").between(0, 100)) &
        (F.col("ad_spend") >= 0) &
        (F.col("unit_price") >= 0) &
        (F.col("revenue") >= 0) &
        (F.abs(F.col("revenue") - expected_revenue) <= 0.01)
    )

    # Add date features
    .withColumn("year", F.year("interaction_date"))
    .withColumn("month", F.month("interaction_date"))
    .withColumn(
        "quarter",
        F.concat(F.lit("Q"), F.quarter("interaction_date"))
    )

    # Add business features
    .withColumn(
        "converted",
        F.when(F.col("orders") > 0, 1).otherwise(0)
    )
    .withColumn(
        "discounted_unit_price",
        F.round(
            F.col("unit_price")
            * (1 - F.col("discount_percent") / 100),
            2
        )
    )
)

display(silver_df.limit(10))

ad_spend,campaign_id,clicks,customer_age,customer_gender,customer_id,customer_region,device_type,discount_percent,impressions,interaction_date,marketing_channel,orders,product_category,product_id,revenue,unit_price,units_sold,website_visits,ingestion_timestamp,source_name,load_date,year,month,quarter,converted,discounted_unit_price
33.64,CMP001,0,48,Male,C0655,Balochistan,Tablet,25.0,4,2025-05-21,Email,0,Electronics,P001,0.0,50000.0,0,0,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,5,Q2,0,37500.0
56.74,CMP001,1,57,Male,C0914,Balochistan,Mobile,20.0,7,2025-10-30,Email,0,Accessories,P005,0.0,2500.0,0,1,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,10,Q4,0,2000.0
67.82,CMP004,0,38,Female,C0734,Punjab,Mobile,30.0,8,2025-04-23,Display,0,Accessories,P005,0.0,2500.0,0,0,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,4,Q2,0,1750.0
52.07,CMP003,0,37,Male,C0715,Khyber Pakhtunkhwa,Mobile,30.0,3,2025-05-23,Search,0,Beauty,P004,0.0,3000.0,0,0,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,5,Q2,0,2100.0
201.08,CMP003,0,20,Female,C0390,Balochistan,Tablet,0.0,10,2025-06-26,Search,0,Electronics,P001,0.0,50000.0,0,0,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,6,Q2,0,50000.0
57.71,CMP001,0,26,Male,C0128,Khyber Pakhtunkhwa,Desktop,20.0,5,2025-10-10,Email,0,Beauty,P004,0.0,3000.0,0,0,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,10,Q4,0,2400.0
45.69,CMP001,0,49,Female,C0592,Khyber Pakhtunkhwa,Mobile,0.0,4,2025-01-24,Email,0,Clothing,P002,0.0,5000.0,0,0,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,1,Q1,0,5000.0
35.03,CMP004,0,20,Female,C0390,Balochistan,Desktop,10.0,6,2025-11-22,Display,0,Footwear,P003,0.0,8000.0,0,0,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,11,Q4,0,7200.0
46.48,CMP002,0,25,Female,C0719,Balochistan,Tablet,15.0,4,2025-10-01,Social Media,0,Electronics,P001,0.0,50000.0,0,0,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,10,Q4,0,42500.0
62.01,CMP002,0,44,Male,C0705,Balochistan,Mobile,30.0,6,2025-12-17,Social Media,0,Accessories,P005,0.0,2500.0,0,0,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,12,Q4,0,1750.0


**check the row count:**

In [0]:
bronze_count = bronze_df.count()
silver_count = silver_df.count()

print(f"Bronze rows: {bronze_count:,}")
print(f"Silver rows: {silver_count:,}")
print(f"Rows removed: {bronze_count - silver_count:,}")

assert silver_count == 10_000, (
    f"Expected 10,000 Silver rows, but found {silver_count:,}"
)

print("Silver row-count validation passed.")

Bronze rows: 10,000
Silver rows: 10,000
Rows removed: 0
Silver row-count validation passed.


In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("default.silver_marketing_interactions")
)

print("Silver table created successfully.")

Silver table created successfully.


**Validate the saved table**

In [0]:
saved_silver_df = spark.table(
    "default.silver_marketing_interactions"
)

saved_silver_count = saved_silver_df.count()

print(f"Silver DataFrame rows: {silver_count:,}")
print(f"Saved Silver rows:     {saved_silver_count:,}")

assert saved_silver_count == silver_count, (
    "Saved Silver table row count does not match the DataFrame."
)

print("Saved Silver table validation passed.")

Silver DataFrame rows: 10,000
Saved Silver rows:     10,000
Saved Silver table validation passed.


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT campaign_id) AS campaigns,
    COUNT(DISTINCT customer_id) AS customers,
    COUNT(DISTINCT product_id) AS products,
    SUM(orders) AS total_orders,
    SUM(units_sold) AS total_units_sold,
    ROUND(SUM(revenue), 2) AS total_revenue,
    MIN(interaction_date) AS first_date,
    MAX(interaction_date) AS last_date
FROM default.silver_marketing_interactions;

total_rows,campaigns,customers,products,total_orders,total_units_sold,total_revenue,first_date,last_date
10000,4,999,5,169,336,3827575.0,2025-01-01,2025-12-31


In [0]:
%sql
SELECT
    'Bronze' AS layer,
    COUNT(DISTINCT customer_id) AS customers
FROM default.bronze_marketing_interactions

UNION ALL

SELECT
    'Silver' AS layer,
    COUNT(DISTINCT customer_id) AS customers
FROM default.silver_marketing_interactions;

layer,customers
Bronze,999
Silver,999


In [0]:
expected_customers_df = (
    spark.range(1, 1001)
    .select(
        F.format_string("C%04d", F.col("id")).alias("customer_id")
    )
)

actual_customers_df = (
    saved_silver_df
    .select("customer_id")
    .distinct()
)

missing_customers_df = expected_customers_df.join(
    actual_customers_df,
    on="customer_id",
    how="left_anti"
)

display(missing_customers_df)

customer_id
C0241


**Gold campaign metrics**

In [0]:
from pyspark.sql import functions as F

silver_df = spark.table(
    "default.silver_marketing_interactions"
)

gold_campaign_df = (
    silver_df
    .groupBy("campaign_id", "marketing_channel")
    .agg(
        F.count("*").alias("total_interactions"),
        F.sum("impressions").alias("total_impressions"),
        F.sum("clicks").alias("total_clicks"),
        F.sum("website_visits").alias("total_website_visits"),
        F.sum("orders").alias("total_orders"),
        F.sum("units_sold").alias("total_units_sold"),
        F.round(F.sum("revenue"), 2).alias("total_revenue"),
        F.round(F.sum("ad_spend"), 2).alias("total_ad_spend")
    )
    .withColumn(
        "ctr_percent",
        F.when(
            F.col("total_impressions") > 0,
            F.round(
                F.col("total_clicks") /
                F.col("total_impressions") * 100,
                2
            )
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "conversion_rate_percent",
        F.when(
            F.col("total_website_visits") > 0,
            F.round(
                F.col("total_orders") /
                F.col("total_website_visits") * 100,
                2
            )
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "roas",
        F.when(
            F.col("total_ad_spend") > 0,
            F.round(
                F.col("total_revenue") /
                F.col("total_ad_spend"),
                2
            )
        ).otherwise(F.lit(0.0))
    )
    .orderBy("campaign_id")
)

display(gold_campaign_df)

campaign_id,marketing_channel,total_interactions,total_impressions,total_clicks,total_website_visits,total_orders,total_units_sold,total_revenue,total_ad_spend,ctr_percent,conversion_rate_percent,roas
CMP001,Email,2565,14028,256,249,50,104,1287175.0,139907.97,1.82,20.08,9.2
CMP002,Social Media,2497,13836,327,301,36,70,723275.0,180748.81,2.36,11.96,4.0
CMP003,Search,2431,13391,425,397,74,141,1533500.0,247788.11,3.17,18.64,6.19
CMP004,Display,2507,13670,165,136,9,21,283625.0,95866.99,1.21,6.62,2.96


In [0]:
(
    gold_campaign_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("default.gold_campaign_metrics")
)

print("Gold campaign metrics table created successfully.")

Gold campaign metrics table created successfully.


In [0]:
%sql
SELECT
    COUNT(*) AS campaigns,
    SUM(total_interactions) AS interactions,
    SUM(total_orders) AS orders,
    SUM(total_units_sold) AS units_sold,
    ROUND(SUM(total_revenue), 2) AS revenue,
    ROUND(SUM(total_ad_spend), 2) AS ad_spend
FROM default.gold_campaign_metrics;

campaigns,interactions,orders,units_sold,revenue,ad_spend
4,10000,169,336,3827575.0,664311.88


**CTR = clicks ÷ impressions × 100
Conversion rate = orders ÷ website visits × 100
ROAS = revenue ÷ ad spend**

In [0]:
gold_customer_df = (
    silver_df
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("number_of_interactions"),
        F.sum("orders").alias("total_orders"),
        F.sum("units_sold").alias("total_units_purchased"),
        F.round(F.sum("revenue"), 2).alias("total_revenue"),
        F.max("interaction_date").alias("last_interaction_date")
    )
    .withColumn(
        "average_order_value",
        F.when(
            F.col("total_orders") > 0,
            F.round(
                F.col("total_revenue") / F.col("total_orders"),
                2
            )
        ).otherwise(F.lit(0.0))
    )
    .orderBy("customer_id")
)

display(gold_customer_df)

customer_id,number_of_interactions,total_orders,total_units_purchased,total_revenue,last_interaction_date,average_order_value
C0001,10,0,0,0.0,2025-12-07,0.0
C0002,9,0,0,0.0,2025-11-27,0.0
C0003,9,0,0,0.0,2025-12-11,0.0
C0004,12,0,0,0.0,2025-12-22,0.0
C0005,11,0,0,0.0,2025-12-17,0.0
C0006,10,0,0,0.0,2025-11-26,0.0
C0007,13,0,0,0.0,2025-12-19,0.0
C0008,6,0,0,0.0,2025-12-02,0.0
C0009,10,0,0,0.0,2025-12-17,0.0
C0010,11,0,0,0.0,2025-12-20,0.0


**Average order value = total revenue ÷ total orders**

**Save the table:**

In [0]:
(
    gold_customer_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("default.gold_customer_metrics")
)

print("Gold customer metrics table created successfully.")

Gold customer metrics table created successfully.


In [0]:
%sql
SELECT
    COUNT(*) AS customers,
    SUM(number_of_interactions) AS interactions,
    SUM(total_orders) AS orders,
    SUM(total_units_purchased) AS units_purchased,
    ROUND(SUM(total_revenue), 2) AS revenue,
    MIN(last_interaction_date) AS earliest_last_interaction,
    MAX(last_interaction_date) AS latest_last_interaction
FROM default.gold_customer_metrics;

customers,interactions,orders,units_purchased,revenue,earliest_last_interaction,latest_last_interaction
999,10000,169,336,3827575.0,2025-03-29,2025-12-31


**third Gold table: product metrics.**

In [0]:
gold_product_df = (
    silver_df
    .groupBy("product_id", "product_category")
    .agg(
        F.count("*").alias("total_interactions"),
        F.sum("impressions").alias("total_impressions"),
        F.sum("clicks").alias("total_clicks"),
        F.sum("website_visits").alias("total_website_visits"),
        F.sum("orders").alias("total_orders"),
        F.sum("units_sold").alias("total_units_sold"),
        F.round(F.sum("revenue"), 2).alias("total_revenue")
    )
    .withColumn(
        "average_selling_price",
        F.when(
            F.col("total_units_sold") > 0,
            F.round(
                F.col("total_revenue") /
                F.col("total_units_sold"),
                2
            )
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "conversion_rate_percent",
        F.when(
            F.col("total_website_visits") > 0,
            F.round(
                F.col("total_orders") /
                F.col("total_website_visits") * 100,
                2
            )
        ).otherwise(F.lit(0.0))
    )
    .orderBy("product_id")
)

display(gold_product_df)

product_id,product_category,total_interactions,total_impressions,total_clicks,total_website_visits,total_orders,total_units_sold,total_revenue,average_selling_price,conversion_rate_percent
P001,Electronics,2017,11160,230,212,35,67,2855000.0,42611.94,16.51
P002,Clothing,2072,11307,257,238,46,93,373500.0,4016.13,19.33
P003,Footwear,1945,10737,227,213,22,47,312000.0,6638.3,10.33
P004,Beauty,2006,11002,235,214,31,55,133950.0,2435.45,14.49
P005,Accessories,1960,10719,224,206,35,74,153125.0,2069.26,16.99


Average selling price = revenue ÷ units sold

Conversion rate = orders ÷ website visits × 100

In [0]:
(
    gold_product_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("default.gold_product_metrics")
)

print("Gold product metrics table created successfully.")

Gold product metrics table created successfully.


In [0]:
%sql
SELECT
    COUNT(*) AS products,
    SUM(total_interactions) AS interactions,
    SUM(total_orders) AS orders,
    SUM(total_units_sold) AS units_sold,
    ROUND(SUM(total_revenue), 2) AS revenue
FROM default.gold_product_metrics;

products,interactions,orders,units_sold,revenue
5,10000,169,336,3827575.0


****

In [0]:
%sql
SELECT
    'Silver' AS table_name,
    COUNT(*) AS interactions,
    SUM(orders) AS orders,
    SUM(units_sold) AS units_sold,
    ROUND(SUM(revenue), 2) AS revenue
FROM default.silver_marketing_interactions

UNION ALL

SELECT
    'Gold Campaign',
    SUM(total_interactions),
    SUM(total_orders),
    SUM(total_units_sold),
    ROUND(SUM(total_revenue), 2)
FROM default.gold_campaign_metrics

UNION ALL

SELECT
    'Gold Customer',
    SUM(number_of_interactions),
    SUM(total_orders),
    SUM(total_units_purchased),
    ROUND(SUM(total_revenue), 2)
FROM default.gold_customer_metrics

UNION ALL

SELECT
    'Gold Product',
    SUM(total_interactions),
    SUM(total_orders),
    SUM(total_units_sold),
    ROUND(SUM(total_revenue), 2)
FROM default.gold_product_metrics;

table_name,interactions,orders,units_sold,revenue
Silver,10000,169,336,3827575.0
Gold Campaign,10000,169,336,3827575.0
Gold Customer,10000,169,336,3827575.0
Gold Product,10000,169,336,3827575.0


In [0]:
%sql
SELECT
    SUM(
        CASE
            WHEN ABS(
                ctr_percent -
                ROUND(total_clicks * 100.0 / total_impressions, 2)
            ) > 0.01
            THEN 1 ELSE 0
        END
    ) AS incorrect_ctr_rows,

    SUM(
        CASE
            WHEN ABS(
                conversion_rate_percent -
                ROUND(total_orders * 100.0 / total_website_visits, 2)
            ) > 0.01
            THEN 1 ELSE 0
        END
    ) AS incorrect_conversion_rows,

    SUM(
        CASE
            WHEN ABS(
                roas -
                ROUND(total_revenue / total_ad_spend, 2)
            ) > 0.01
            THEN 1 ELSE 0
        END
    ) AS incorrect_roas_rows
FROM default.gold_campaign_metrics;

incorrect_ctr_rows,incorrect_conversion_rows,incorrect_roas_rows
0,0,0


**Create the dashboard Gold table**

In [0]:
gold_dashboard_df = (
    silver_df
    .groupBy(
        "interaction_date",
        "year",
        "month",
        "quarter",
        "campaign_id",
        "marketing_channel",
        "product_id",
        "product_category",
        "customer_region",
        "device_type"
    )
    .agg(
        F.count("*").alias("total_interactions"),
        F.sum("impressions").alias("total_impressions"),
        F.sum("clicks").alias("total_clicks"),
        F.sum("website_visits").alias("total_website_visits"),
        F.sum("orders").alias("total_orders"),
        F.sum("units_sold").alias("total_units_sold"),
        F.round(F.sum("revenue"), 2).alias("total_revenue"),
        F.round(F.sum("ad_spend"), 2).alias("total_ad_spend")
    )
)

display(gold_dashboard_df.limit(10))

interaction_date,year,month,quarter,campaign_id,marketing_channel,product_id,product_category,customer_region,device_type,total_interactions,total_impressions,total_clicks,total_website_visits,total_orders,total_units_sold,total_revenue,total_ad_spend
2025-09-04,2025,9,Q3,CMP004,Display,P004,Beauty,Balochistan,Mobile,1,4,0,0,0,0,0.0,22.63
2025-09-02,2025,9,Q3,CMP003,Search,P001,Electronics,Sindh,Tablet,1,2,0,0,0,0,0.0,36.88
2025-04-07,2025,4,Q2,CMP003,Search,P004,Beauty,Balochistan,Desktop,1,4,0,0,0,0,0.0,83.97
2025-02-25,2025,2,Q1,CMP003,Search,P004,Beauty,Sindh,Tablet,1,4,0,0,0,0,0.0,84.76
2025-06-20,2025,6,Q2,CMP002,Social Media,P005,Accessories,Sindh,Desktop,1,2,0,0,0,0,0.0,31.0
2025-08-17,2025,8,Q3,CMP003,Search,P004,Beauty,Punjab,Desktop,1,1,0,0,0,0,0.0,19.3
2025-12-14,2025,12,Q4,CMP002,Social Media,P005,Accessories,Sindh,Mobile,1,8,0,0,0,0,0.0,101.31
2025-11-10,2025,11,Q4,CMP002,Social Media,P004,Beauty,Punjab,Mobile,1,5,0,0,0,0,0.0,56.01
2025-08-09,2025,8,Q3,CMP001,Email,P001,Electronics,Balochistan,Desktop,1,3,0,0,0,0,0.0,28.39
2025-03-13,2025,3,Q1,CMP002,Social Media,P001,Electronics,Sindh,Desktop,1,8,0,0,0,0,0.0,80.33


In [0]:
(
    gold_dashboard_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("default.gold_dashboard_metrics")
)

print("Gold dashboard metrics table created successfully.")

Gold dashboard metrics table created successfully.


In [0]:
%sql
SELECT
    SUM(total_interactions) AS interactions,
    SUM(total_orders) AS orders,
    SUM(total_units_sold) AS units_sold,
    ROUND(SUM(total_revenue), 2) AS revenue,
    ROUND(SUM(total_ad_spend), 2) AS ad_spend
FROM default.gold_dashboard_metrics;

interactions,orders,units_sold,revenue,ad_spend
10000,169,336,3827575.0,664311.88
